## Step 1: Reorganize Dictionary Entries

This step reads the raw dictionary-style lexical table, cleans the XML/HTML-like definition field, creates the `clean1` text field, and splits each entry into `headword`, `pos`, `sense1`, and `example1` rows. Change every `CHANGE_THIS_TO_YOUR_PATH/...` placeholder before running.



In [1]:
import os
import re
import pandas as pd

#
INPUT_PATH     = r"CHANGE_THIS_TO_YOUR_PATH/test/现汉7_entries_two_chars_cleaned_separated_monoseme_psycho_cleaned.csv"
OUTPUT_CLEAN1  = r"CHANGE_THIS_TO_YOUR_PATH/test/monoseme_cleaned_output.xlsx"
OUTPUT_SPLIT   = r"CHANGE_THIS_TO_YOUR_PATH/test/monoseme_pos_sense_example.xlsx"


COL_DEFINITION     = "definition"
COL_HEADWORD_INDEX = 1
POS_LABEL_SAMEAS   = "同义"

# POS parsing and normalization helper.
HEADWORD_POS_OVERRIDES = {
    "当时": {1: "动词"},
}

# =========================
# =========================
RE_DEF_BLOCKS = re.compile(r"<\s*def\s*>(.*?)<\s*/\s*def\s*>", re.IGNORECASE | re.DOTALL)
RE_NUM_IN_DEF = re.compile(r"<\s*num\s*>(.*?)<\s*/\s*num\s*>", re.IGNORECASE | re.DOTALL)
RE_PS_IN_DEF  = re.compile(r"<\s*ps\s*>(.*?)<\s*/\s*ps\s*>",  re.IGNORECASE | re.DOTALL)
RE_SUP_BLOCK  = re.compile(r"<\s*sup\s*>.*?<\s*/\s*sup\s*>",  re.IGNORECASE | re.DOTALL)
RE_HTML_TAG   = re.compile(r"</?[^>]+>")
RE_CUTOFF_ELSE= re.compile(r"另见.*", re.DOTALL)
RE_FULLWIDTH_ANGLE_GROUPS = re.compile(r"〈([^〉]*)〉")
RE_CN_QUOTES  = re.compile(r"“([^”]*)”")

ALLOWED_CLASS = (
    r"\u4E00-\u9FFF"
    r"\u3400-\u4DBF"
    r"\u3000-\u303F"
    r"\uFF00-\uFFEF"
    r"\u2776-\u2793"  # ❶❷…
    r"\u201C\u201D"   # “ ”
    r"\u25C7"         # ◇
)
RE_DISALLOWED = re.compile(fr"[^{ALLOWED_CLASS}\s]")

RE_PAREN_NUM_FW  = re.compile(r"（[0-9\uFF10-\uFF19]+）")
RE_PAREN_NUM_ASC = re.compile(r"\([0-9]+\)")
RE_EMPTY_PARENS  = re.compile(r"(（\s*）|\(\s*\))")

RE_BULLET          = re.compile(r"[\u2776-\u2793]")
RE_SPLIT_BY_BULLET = re.compile(r"(?=[\u2776-\u2793])")
RE_LEADING_POS     = re.compile(r"^\s*pos\s*:\s*([^\s;；：:]+)\s*[;；]\s*", flags=re.IGNORECASE)
RE_SEG_POS         = re.compile(r"^\s*pos\s*:\s*([^\s;；：:]+)\s*[;；]\s*", flags=re.IGNORECASE)
SEGMENT_RE_NO_BULLET = re.compile(
    r"(?:pos\s*:\s*([^\s;；：:]+)\s*[;；]\s*)(.*?)"
    r"(?=(?:pos\s*:)|$)",
    flags=re.IGNORECASE | re.DOTALL
)

CATEGORY_POS_RE = re.compile(r"^([\u4E00-\u9FFF]{1,8}词)[。．]\s*")
SAMEAS_STRICT_RE = re.compile(r'^同[“”「」『』《》〈〉](?P<inner>.+)[“”「」『』《》〈〉][。．]?$')

RE_DIA_EX = re.compile(r"◇\s*<\s*ex\s*>(.*?)<\s*/\s*ex\s*>", re.IGNORECASE | re.DOTALL)

def read_any(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path.lower())[1]
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if ext in [".csv", ".tsv"]:
        sep = "," if ext == ".csv" else "\t"
        return pd.read_csv(path, sep=sep, encoding="utf-8-sig")
    raise ValueError("Only .xlsx/.xls/.csv/.tsv input files are supported.")

def write_any(df: pd.DataFrame, path: str):
    ext = os.path.splitext(path.lower())[1]
    if ext in [".xlsx", ".xls"]:
        try:
            with pd.ExcelWriter(path, engine="openpyxl") as writer:
                df.to_excel(writer, index=False)
        except Exception:
            df.to_excel(path, index=False)
    elif ext in [".csv", ".tsv"]:
        sep = "," if ext == ".csv" else "\t"
        df.to_csv(path, index=False, encoding="utf-8-sig", sep=sep)
    else:
        raise ValueError("Only .xlsx/.xls/.csv/.tsv output files are supported.")

def keep_allowed_chars(text: str) -> str:
    return RE_DISALLOWED.sub("", text or "")

def preserve_fullwidth_angles_with_han(text: str) -> str:
    def repl(m):
        inner = m.group(1)
        if re.search(r"[\u4E00-\u9FFF\u3400-\u4DBF]", inner or ""):
            return f"〈{inner}〉"
        return ""
    return RE_FULLWIDTH_ANGLE_GROUPS.sub(repl, text or "")

ALLOWED_IN_CN_QUOTES = re.compile(
    r'^[\u4E00-\u9FFF\u3400-\u4DBF'
    r'\u3000-\u303F'
    r'\uFF00-\uFFEF'
    r'\s]+$'
)
def preserve_cn_quotes_with_han(text: str) -> str:
    def repl(m):
        inner = m.group(1)
        if ALLOWED_IN_CN_QUOTES.match(inner or ""):
            return f"“{inner}”"
        return ""
    return RE_CN_QUOTES.sub(repl, text or "")

def normalize_space(s: str) -> str:
    if s is None: return ""
    return re.sub(r"\s+", " ", s).strip()

def strip_end_english_period(s: str) -> str:
    if s is None: return ""
    s = s.strip()
    s = re.sub(r"\.+$", "", s)
    return s.strip()

def ensure_cn_period(s: str) -> str:
    if not s:
        return s
    if re.search(r"[。！？!?]$", s):
        return s
    return s + "。"

def protect_paren_numbers(text: str):
    repls = []
    def _repl(m):
        token = f"__PAREN_NUM_{len(repls)}__"
        repls.append((token, m.group(0)))
        return token
    t = RE_PAREN_NUM_FW.sub(_repl, text or "")
    t = RE_PAREN_NUM_ASC.sub(_repl, t)
    return t, repls

def restore_paren_numbers(text: str, repls):
    for token, orig in repls:
        text = text.replace(token, orig)
    return text

# Generate or process the clean1 text field.
def extract_ps_head(block: str) -> str:
    m = RE_PS_IN_DEF.search(block or "")
    if not m:
        return ""
    inner = m.group(1)
    chars = re.findall(r"[\u4E00-\u9FFF\u3400-\u4DBF]+", inner)
    return chars[0] if chars else ""

def _clean_text_block(raw: str, headword: str) -> str:
    """Apply standard cleaning to a raw definition segment: remove HTML tags, replace the headword marker, and keep only allowed characters."""
    if raw is None:
        raw = ""
    s = RE_SUP_BLOCK.sub("", raw)
    s = preserve_fullwidth_angles_with_han(s)
    s = RE_HTML_TAG.sub("", s)
    s = preserve_cn_quotes_with_han(s)
    t, saved = protect_paren_numbers(s)
    t = keep_allowed_chars(t)
    if headword:
        t = t.replace("～", str(headword))
    t = restore_paren_numbers(t, saved)
    return normalize_space(t)

def clean_def_block(block: str, headword: str) -> str:
    """
    Build one clean1 segment from a raw definition block.
    Key point: before removing <num>/<ps>, use RE_DIA_EX to split the special example block into sense and example parts.
    """
    num_m   = RE_NUM_IN_DEF.search(block)
    num_txt = keep_allowed_chars(num_m.group(1)) if num_m else ""
    ps_head = extract_ps_head(block)

    m = RE_DIA_EX.search(block or "")
    if m:
        sense_raw = block[:m.start()]
        ex_raw    = m.group(1)
    else:
        sense_raw = block
        ex_raw    = ""

    sense_raw = RE_NUM_IN_DEF.sub("", sense_raw)
    sense_raw = RE_PS_IN_DEF.sub("",  sense_raw)

    sense_clean = _clean_text_block(sense_raw, headword)
    ex_clean    = _clean_text_block(ex_raw,   headword)

    # Generate or process the clean1 text field.
    seg = ""
    if num_txt:
        seg += num_txt
    if ps_head:
        seg += f"pos:{ps_head}; "
    seg += sense_clean
    if ex_clean:
        seg += "：" + ex_clean
    return seg.strip()

def build_clean1(text: str, headword: str) -> str:
    s = str(text) if text is not None else ""
    s = RE_SUP_BLOCK.sub("", s)
    s = RE_CUTOFF_ELSE.sub("", s)

    def_blocks = RE_DEF_BLOCKS.findall(s)
    if not def_blocks:
        def_blocks = [s]

    segs = [clean_def_block(b, headword) for b in def_blocks if b and b.strip()]
    return "".join(segs).strip()

# Generate or process the clean1 text field.
def clean_text_keep_rules(s: str) -> str:
    s = normalize_space(s)
    s = RE_EMPTY_PARENS.sub("", s)
    s = preserve_cn_quotes_with_han(s)
    return s.strip()

def sentence_split_cn(text: str):
    if not text:
        return []
    parts = re.findall(r"[^。！？!?]+[。！？!?]?", text)
    return [p.strip() for p in parts if p and p.strip()]

def split_examples(ex_str: str):
    if not ex_str:
        return [""]
    s = ex_str.replace("◇", "。")
    frags = re.split(r"[｜\|]+", s)
    out = []
    for f in frags:
        f = clean_text_keep_rules(f)
        if not f:
            continue
        for sent in sentence_split_cn(f):
            sent = strip_end_english_period(sent)
            if not sent:
                continue
            sent = ensure_cn_period(sent)
            out.append(sent)
    seen, uniq = set(), []
    for t in out:
        if t not in seen:
            seen.add(t)
            uniq.append(t)
    return uniq or [""]

def prepend_prefix(prefix: str, sense: str) -> str:
    if not prefix:
        return sense
    if sense.startswith(prefix):
        return sense
    return (prefix + sense).strip()

def is_pure_paren_variant(sense: str, headword: str) -> bool:
    if not sense:
        return False
    s = sense.replace(" ", "")
    if not (s.startswith("（") and s.endswith("）")):
        return False
    inner = s[1:-1]
    if not headword or headword not in inner:
        return False
    if any(ch in inner for ch in ["：", "；", "。", "｜"]):
        return False
    try:
        return len(inner) <= len(headword) + 4
    except Exception:
        return True

def merge_pos(base_pos: str, extra_pos: str) -> str:
    items = []
    for chunk in (base_pos, extra_pos):
        if chunk:
            for p in re.split(r"[；;]\s*", str(chunk).strip()):
                p = p.strip()
                if p and p not in items:
                    items.append(p)
    return "；".join(items)

def extract_category_from_sense(sense: str):
    m = CATEGORY_POS_RE.match(sense)
    if not m:
        return "", sense
    cat = m.group(1)
    new_sense = sense[m.end():].lstrip()
    return cat, new_sense

def apply_sameas_pos_mark_strict(pos: str, sense: str):
    s0 = sense.strip()
    if SAMEAS_STRICT_RE.match(s0):
        return POS_LABEL_SAMEAS
    return pos

# POS parsing and normalization helper.
POS_CANON_MAP = {
    "名": "名词", "名词": "名词",
    "动": "动词", "動": "动词", "动词": "动词",
    "形": "形容词", "形容": "形容词", "形容詞": "形容词", "形容词": "形容词",
    "代": "代词", "副": "副词", "连": "连词", "連": "连词",
    "介": "介词", "助": "助词", "叹": "叹词", "嘆": "叹词",
    "拟声": "拟声词", "擬聲": "拟声词",
    "语气": "语气词", "語氣": "语气词",
    "区别": "区别词", "區別": "区别词",
    "量": "量词", "名量": "名量词",
    "时量": "时量词", "時量": "时量词",
    "数": "数词", "數": "数词",
}
def normalize_pos_tokens(pos: str) -> str:
    if not pos:
        return pos
    toks = [t.strip() for t in re.split(r"[；;]+", pos) if t.strip()]
    norm, seen = [], set()
    for t in toks:
        if t == POS_LABEL_SAMEAS:
            t2 = t
        else:
            t2 = POS_CANON_MAP.get(t, t)
            if t2 and t2 != POS_LABEL_SAMEAS and not t2.endswith("词"):
                if re.fullmatch(r"[\u4E00-\u9FFF\u3400-\u4DBF]+", t2):
                    t2 = t2 + "词"
        if t2 not in seen:
            seen.add(t2)
            norm.append(t2)
    return "；".join(norm)

# POS parsing and normalization helper.
KEEP_EARLY_CAT_IN_POS = {"属性词", "数量词"}

def extract_domain_prefix_before_first_bullet(s: str, after_leading_pos_end: int = 0):
    fb = RE_BULLET.search(s)
    if not fb:
        return "", None
    pre = s[after_leading_pos_end:fb.start()]
    pre = clean_text_keep_rules(pre)
    if pre and re.fullmatch(r"(?:\s*〈[^〉]+〉\s*)+", pre):
        return pre, fb.start()
    return "", fb.start()

def extract_global_category_before_bullet(s: str, after_leading_pos_end: int = 0):
    fb = RE_BULLET.search(s)
    if not fb:
        return ""
    pre = s[after_leading_pos_end:fb.start()]
    pre = clean_text_keep_rules(pre)
    m = CATEGORY_POS_RE.match(pre)
    return m.group(1) if m else ""

def strip_leading_category(text: str) -> str:
    if not text:
        return text
    m = CATEGORY_POS_RE.match(text)
    if m:
        return text[m.end():].lstrip()
    return text

def override_core_pos_if_needed(headword: str, sense_idx: int, pos: str) -> str:
    mapping = HEADWORD_POS_OVERRIDES.get(headword or "", {})
    override_core = mapping.get(sense_idx)
    if not override_core:
        return normalize_pos_tokens(pos)
    if not pos:
        return override_core
    toks = [t.strip() for t in re.split(r"[；;]+", pos) if t.strip()]
    cats = [t for t in toks if t != POS_LABEL_SAMEAS and t not in {"名词", "动词", "形容词", "代词", "副词"}]
    new_pos = "；".join(([override_core] + cats)) if cats else override_core
    return normalize_pos_tokens(new_pos)

def parse_segment_with_pos_inheritance(seg: str,
                                       inherited_pos: str,
                                       prefix_text: str,
                                       headword: str,
                                       global_cat: str,
                                       sense_idx: int):
    s = seg.strip()
    s = re.sub(r"^[\u2776-\u2793]\s*", "", s)

    mpos = RE_SEG_POS.match(s)
    pos_source_is_explicit = bool(mpos)

    if mpos:
        pos = clean_text_keep_rules(mpos.group(1))
        body = s[mpos.end():]
        current_pos = pos
    else:
        pos = inherited_pos or ""
        body = s
        current_pos = inherited_pos

    # Sense-level processing rule.
    if "：" in body:
        sense, ex = body.split("：", 1)
    elif ":" in body:
        sense, ex = body.split(":", 1)
    else:
        sense, ex = body, ""

    sense = clean_text_keep_rules(sense)
    ex    = clean_text_keep_rules(ex)

    if (not pos) and is_pure_paren_variant(sense, headword):
        return [], current_pos

    # Sense-level processing rule.
    prefix_text = strip_leading_category(prefix_text)
    sense = prepend_prefix(prefix_text, sense)

    early_cat, sense_wo_early = extract_category_from_sense(sense)

    # POS parsing and normalization helper.
    pos = merge_pos(pos, global_cat)

    if early_cat:
        if early_cat in KEEP_EARLY_CAT_IN_POS:
            if pos_source_is_explicit:
                pos = merge_pos(pos, early_cat)
            else:
                pos = early_cat
            sense = sense_wo_early
        else:
            sense = f"{early_cat}；{sense_wo_early}"
    else:
        sense = sense_wo_early

    # POS parsing and normalization helper.
    pos = apply_sameas_pos_mark_strict(pos, sense)

    # POS parsing and normalization helper.
    pos = normalize_pos_tokens(pos)

    pos = override_core_pos_if_needed(headword, sense_idx, pos)

    examples = split_examples(ex)

    # Sense-level processing rule.
    sense = strip_end_english_period(sense)
    sense = ensure_cn_period(sense)

    rows = [{"pos": pos, "sense1": sense, "example1": e} for e in examples]
    return rows, current_pos

def parse_clean1_to_rows(text: str, headword: str):
    s = str(text) if text is not None else ""
    s = s.strip()
    if not s:
        return []

    records = []

    if RE_BULLET.search(s):
        inherited_pos = ""
        prefix_from_pos = ""
        leading_pos_end = 0

        mhead = RE_LEADING_POS.match(s)
        fb = RE_BULLET.search(s)
        if mhead and fb and mhead.end() <= fb.start():
            inherited_pos = clean_text_keep_rules(mhead.group(1))
            prefix_raw = s[mhead.end():fb.start()]
            prefix_from_pos = strip_leading_category(clean_text_keep_rules(prefix_raw))
            leading_pos_end = mhead.end()
        else:
            inherited_pos = ""
            prefix_from_pos = ""
            leading_pos_end = 0

        domain_prefix, fb_start = extract_domain_prefix_before_first_bullet(s, after_leading_pos_end=leading_pos_end)
        if fb_start is None:
            fb_start = fb.start()
        global_cat = extract_global_category_before_bullet(s, after_leading_pos_end=leading_pos_end)
        full_prefix = (prefix_from_pos + domain_prefix).strip()

        s_after = s[fb_start:]
        chunks = [c for c in RE_SPLIT_BY_BULLET.split(s_after) if c.strip()]

        sense_idx = 0
        for seg in chunks:
            if not seg.strip():
                continue
            sense_idx += 1
            rows, inherited_pos = parse_segment_with_pos_inheritance(
                seg, inherited_pos, full_prefix, headword, global_cat, sense_idx
            )
            for r in rows:
                if r["sense1"] == "" and r["example1"] == "":
                    continue
                records.append(r)
    else:
        any_hit = False
        for m in SEGMENT_RE_NO_BULLET.finditer(s):
            any_hit = True
            pos  = clean_text_keep_rules(m.group(1))
            body = m.group(2) or ""
            if "：" in body:
                sense, ex = body.split("：", 1)
            elif ":" in body:
                sense, ex = body.split(":", 1)
            else:
                sense, ex = body, ""

            sense = clean_text_keep_rules(sense)
            ex    = clean_text_keep_rules(ex)

            cat, sense = extract_category_from_sense(sense)
            if cat:
                pos = merge_pos(pos, cat)
            pos = apply_sameas_pos_mark_strict(pos, sense)
            pos = normalize_pos_tokens(pos)

            examples = split_examples(ex)

            sense = strip_end_english_period(sense)
            sense = ensure_cn_period(sense)

            for e in examples:
                records.append({"pos": pos, "sense1": clean_text_keep_rules(sense), "example1": e})
        if not any_hit:
            cat0, sense0 = extract_category_from_sense(s)
            pos0 = merge_pos("", cat0)
            pos0 = apply_sameas_pos_mark_strict(pos0, sense0)
            pos0 = normalize_pos_tokens(pos0)

            sense0 = strip_end_english_period(sense0)
            sense0 = ensure_cn_period(sense0)

            records.append({"pos": pos0, "sense1": clean_text_keep_rules(sense0), "example1": ""})

    return records

# =========================
# =========================
def main():
    df = read_any(INPUT_PATH)

    if COL_DEFINITION not in df.columns:
        raise KeyError(f"Could not find the configured definition column: {COL_DEFINITION}")
    if COL_HEADWORD_INDEX >= len(df.columns):
        raise IndexError("Column-B index is out of range; please check COL_HEADWORD_INDEX.")

    head_col = df.columns[COL_HEADWORD_INDEX]

    # Generate or process the clean1 text field.
    df["clean1"] = [
        build_clean1(row.get(COL_DEFINITION, ""), str(row.get(head_col, "") or ""))
        for _, row in df.iterrows()
    ]
    cols = df.columns.tolist()
    cols.remove("clean1")
    insert_at = cols.index(COL_DEFINITION) + 1
    df_clean1 = df[cols[:insert_at] + ["clean1"] + cols[insert_at:]]
    write_any(df_clean1, OUTPUT_CLEAN1)

    exploded_rows = []
    for _, row in df_clean1.iterrows():
        base = row.to_dict()
        headword = str(base.get(head_col, "") or "")
        parsed = parse_clean1_to_rows(base.get("clean1", ""), headword)
        for rec in parsed:
            new_row = base.copy()
            new_row.update(rec)
            exploded_rows.append(new_row)
    out = pd.DataFrame(exploded_rows)

    # Generate or process the clean1 text field.
    cols2 = out.columns.tolist()
    for c in ["pos", "sense1", "example1"]:
        if c in cols2:
            cols2.remove(c)
    insert_at2 = cols2.index("clean1") + 1
    new_cols = cols2[:insert_at2] + ["pos", "sense1", "example1"] + cols2[insert_at2:]
    out = out[new_cols]

    out["pos"]      = out["pos"].astype(str).str.strip()
    out["sense1"]   = out["sense1"].astype(str).str.strip()
    out["example1"] = out["example1"].astype(str).str.strip()
    out[head_col]   = out[head_col].astype(str).str.strip()

    bad_mask = out["example1"].str.fullmatch(r"\s*［英］[。．]?\s*")
    out = out[~bad_mask].reset_index(drop=True)

    # Example-sentence processing rule.
    # out = out[out["example1"].str.strip() != ""].reset_index(drop=True)

    before = len(out)
    out = out.drop_duplicates(subset=[head_col, "pos", "sense1", "example1"], keep="first").reset_index(drop=True)
    after = len(out)

    write_any(out, OUTPUT_SPLIT)

    print("\n=== clean1 preview ===")
    print(df_clean1[[head_col, "clean1"]].head(5))
    print("\n=== Split-table preview (before/after deduplication: %d -> %d) ===" % (before, after))
    print(out[[head_col, "pos", "sense1", "example1"]].head(10))
    print(f"\nCompleted:\n 1) clean1 -> {OUTPUT_CLEAN1}\n 2) split table after deduplication -> {OUTPUT_SPLIT}")

if __name__ == "__main__":
    main()



=== clean1 preview ===

[Preview table omitted here: source-language lexical content will appear when you run this cell on your data.]

=== Split-table preview (before/after deduplication: 8826 -> 8754) ===

[Preview table omitted here: source-language headwords, senses, and examples will appear when you run this cell on your data.]

Completed:
 1) clean1 -> CHANGE_THIS_TO_YOUR_PATH/test/monoseme_cleaned_output.xlsx
 2) split table after deduplication -> CHANGE_THIS_TO_YOUR_PATH/test/monoseme_pos_sense_example.xlsx


## Step 2: Clean the Organized Homonym, Polyseme, and Monoseme Tables

This step reads the three organized `*_pos_sense_example.xlsx` tables, removes dialect-tagged and same-as/synonym rows, optionally removes rows with blank examples, applies whole-word single-sense filtering where appropriate, and writes cleaned/deleted sheets for each lexical group.


In [2]:
import os
import pandas as pd
import re

INPUT_HOM  = r"CHANGE_THIS_TO_YOUR_PATH/test/homonym_pos_sense_example.xlsx"
INPUT_POLY = r"CHANGE_THIS_TO_YOUR_PATH/test/polyseme_pos_sense_example.xlsx"
INPUT_MONO = r"CHANGE_THIS_TO_YOUR_PATH/test/monoseme_pos_sense_example.xlsx"

OUTPUT_HOM = r"CHANGE_THIS_TO_YOUR_PATH/other_svaed_new/01_organization/homonym_pos_sense_example_cleaned.xlsx"
OUTPUT_POLY= r"CHANGE_THIS_TO_YOUR_PATH/other_svaed_new/01_organization/polyseme_pos_sense_example_cleaned.xlsx"
OUTPUT_MONO= r"CHANGE_THIS_TO_YOUR_PATH/other_svaed_new/01_organization/monoseme_pos_sense_example_cleaned.xlsx"

os.makedirs(os.path.dirname(OUTPUT_HOM), exist_ok=True)

def read_any(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path.lower())[1]
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if ext in [".csv", ".tsv"]:
        sep = "," if ext == ".csv" else "\t"
        return pd.read_csv(path, sep=sep, encoding="utf-8-sig")
    raise ValueError("Only .xlsx/.xls/.csv/.tsv input files are supported.")

def write_with_deleted(keep_df: pd.DataFrame,
                       drop_df: pd.DataFrame,
                       out_path: str):
    """
    Write one Excel file with two sheets: cleaned rows and deleted rows.
    """
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        keep_df.to_excel(writer, sheet_name="cleaned", index=False)
        drop_df.to_excel(writer, sheet_name="delete_items", index=False)

########################################
########################################
def drop_single_sense_words(df_after_first_round: pd.DataFrame,
                            original_drop_df: pd.DataFrame,
                            file_label: str,
                            enable_single_sense_filter: bool):
    """
    After the first filtering round, optionally remove whole headwords that are left with only one unique sense.
    Return the retained rows, deleted rows, and debug information for logging.
    """

    kept_df_initial = df_after_first_round.copy()
    dropped_df_initial = original_drop_df.copy()

    if not enable_single_sense_filter:
        debug_info = {
            "single_sense_heads": [],
            "dropped_rows_after_second_round": 0
        }
        return kept_df_initial, dropped_df_initial, debug_info

    # Sense-level processing rule.
    if ("headword" not in kept_df_initial.columns) or ("sense1" not in kept_df_initial.columns):
        debug_info = {
            "single_sense_heads": [],
            "dropped_rows_after_second_round": 0,
            "warning": f"[{file_label}] Missing headword or sense1; skipping single-sense whole-word filtering"
        }
        print(f"\n[{file_label}] Warning: headword or sense1 column was not found; skipping single-sense whole-word filtering.")
        return kept_df_initial, dropped_df_initial, debug_info

    # Sense-level processing rule.
    sense_counts = (
        kept_df_initial
        .assign(
            _hw    = kept_df_initial["headword"].astype(str),
            _sense = kept_df_initial["sense1"].astype(str)
        )
        .groupby("_hw")["_sense"]
        .nunique(dropna=True)
        .reset_index()
        .rename(columns={"_hw": "headword", "_sense": "unique_sense_count"})
    )

    # Sense-level processing rule.
    single_sense_heads = sense_counts.loc[
        sense_counts["unique_sense_count"] == 1, "headword"
    ].tolist()

    mask_singleton_delete = kept_df_initial["headword"].astype(str).isin(single_sense_heads)

    dropped_df_singleton = kept_df_initial[mask_singleton_delete].copy()
    kept_df_final        = kept_df_initial[~mask_singleton_delete].copy()

    dropped_df_full = pd.concat(
        [dropped_df_initial, dropped_df_singleton],
        ignore_index=True
    )

    debug_info = {
        "single_sense_heads": single_sense_heads,
        "dropped_rows_after_second_round": len(dropped_df_singleton)
    }

    return kept_df_final, dropped_df_full, debug_info

########################################
########################################
def clean_one_file(df: pd.DataFrame,
                   file_label: str,
                   apply_rule_example_blank: bool,
                   apply_single_sense_filter: bool):
    """
    Clean one lexical table and return the retained rows plus the deleted rows.
    The flags control blank-example filtering and whole-word single-sense filtering.
    """

    needed_cols = ["sense1", "pos"]
    if apply_rule_example_blank:
        needed_cols.append("example1")

    missing = [c for c in needed_cols if c not in df.columns]
    if missing:
        raise KeyError(f"[{file_label}] Missing required columns: {missing}; current columns: {list(df.columns)}")

    # Sense-level processing rule.
    sense1_str = df["sense1"].astype(str).fillna("")
    mask_fang = sense1_str.str.contains("〈方〉", na=False)

    pos_str = df["pos"].astype(str)
    mask_tongyi = pos_str.str.strip().eq("同义")

    # Example-sentence processing rule.
    if apply_rule_example_blank:
        ex_series = df["example1"]
        mask_blank_example = ex_series.isna() | ex_series.astype(str).str.strip().eq("")
    else:
        mask_blank_example = pd.Series([False]*len(df), index=df.index)

    mask_drop_initial = mask_fang | mask_tongyi | mask_blank_example

    dropped_df_initial = df[mask_drop_initial].copy()
    kept_df_initial    = df[~mask_drop_initial].copy()

    # Sense-level processing rule.
    kept_df_final, dropped_df_full, dbg = drop_single_sense_words(
        kept_df_initial,
        dropped_df_initial,
        file_label=file_label,
        enable_single_sense_filter=apply_single_sense_filter
    )

    total_before = len(df)
    drop_fang = int(mask_fang.sum())
    drop_tongyi = int(mask_tongyi.sum())
    drop_blank = int(mask_blank_example.sum())
    drop_union_first = int(mask_drop_initial.sum())
    drop_union_after = len(dropped_df_full)

    print(f"\n=== {file_label} cleaning summary ===")
    print(f"input row count: {total_before}")
    print(f"rows removed because sense1 contains ???: {drop_fang}")
    print(f"rows removed because pos == '??': {drop_tongyi}")
    if apply_rule_example_blank:
        print(f"rows removed because example1 is blank: {drop_blank}")
    else:
        print("rows removed because example1 is blank: not applicable (not checked)")

    print(f"rows removed in the first filtering round: {drop_union_first}")
    if apply_single_sense_filter:
        print(f"additional rows removed in the second filtering round (whole word left with only one sense): {dbg['dropped_rows_after_second_round']}")
        print(f"number of headwords removed as whole words: {len(dbg['single_sense_heads'])}")
        if dbg['single_sense_heads']:
            print("first 10 example headwords removed as whole words:",
                  ", ".join(dbg['single_sense_heads'][:10]))
    print(f"final removed row count: {drop_union_after}")
    print(f"final output row count: {len(kept_df_final)}")

    if "headword" in df.columns:
        original_heads   = set(df["headword"].dropna().astype(str).unique())
        remaining_heads  = set(kept_df_final["headword"].dropna().astype(str).unique())
        removed_heads    = sorted(original_heads - remaining_heads)

        print(f"original headword count: {len(original_heads)}")
        print(f"retained headword count: {len(remaining_heads)}")
        print(f"fully removed headword count: {len(removed_heads)}")
        if removed_heads:
            print("first 10 fully removed headwords:", ", ".join(removed_heads[:10]))

    return kept_df_final, dropped_df_full

########################################
########################################
def main():
    df_hom  = read_any(INPUT_HOM)
    df_poly = read_any(INPUT_POLY)
    df_mono = read_any(INPUT_MONO)

    # homonym:
    # Example-sentence processing rule.
    hom_keep, hom_drop = clean_one_file(
        df_hom,
        file_label="homonym",
        apply_rule_example_blank=False,
        apply_single_sense_filter=True
    )

    if ("headword" in hom_keep.columns) and ("definition" in hom_keep.columns):
        def_counts = (
            hom_keep
            .assign(
                _hw  = hom_keep["headword"].astype(str),
                _def = hom_keep["definition"].astype(str)
            )
            .groupby("_hw")["_def"]
            .nunique(dropna=True)
            .reset_index()
        )

        single_def_heads = def_counts.loc[def_counts["_def"] == 1, "_hw"].tolist()

        if single_def_heads:
            mask_single_def = hom_keep["headword"].astype(str).isin(single_def_heads)
            hom_drop_def = hom_keep[mask_single_def].copy()
            hom_keep = hom_keep[~mask_single_def].copy()
            hom_drop = pd.concat([hom_drop, hom_drop_def], ignore_index=True)

            print("\n=== additional homonym definition-level filtering summary ===")
            print(f"number of headwords with only one unique definition entry: {len(single_def_heads)}")
            print("first 10 example headwords:", ", ".join(single_def_heads[:10]))
            print(f"rows removed because the whole headword had only one unique definition entry: {len(hom_drop_def)}")
            print(f"remaining homonym rows after definition-level filtering: {len(hom_keep)}")
        else:
            print("\n=== additional homonym definition-level filtering summary ===")
            print("No headword with only one unique definition entry was found.")
    else:
        print("\n[homonym] Warning: headword or definition column is missing, so definition-based whole-word filtering cannot be applied.")

    # polyseme:
    # Example-sentence processing rule.
    poly_keep, poly_drop = clean_one_file(
        df_poly,
        file_label="polyseme",
        apply_rule_example_blank=True,
        apply_single_sense_filter=True
    )

    # monoseme:
    # Example-sentence processing rule.
    mono_keep, mono_drop = clean_one_file(
        df_mono,
        file_label="monoseme",
        apply_rule_example_blank=True,
        apply_single_sense_filter=False
    )

    write_with_deleted(hom_keep, hom_drop, OUTPUT_HOM)
    write_with_deleted(poly_keep, poly_drop, OUTPUT_POLY)
    write_with_deleted(mono_keep, mono_drop, OUTPUT_MONO)

    print("\nAll done. Output files:")
    print("  ", OUTPUT_HOM)
    print("  ", OUTPUT_POLY)
    print("  ", OUTPUT_MONO)

if __name__ == "__main__":
    main()


=== homonym cleaning summary ===
input row count: 483
rows removed because sense1 contains ???: 14
rows removed because pos == '??': 4
rows removed because example1 is blank: not applicable (not checked)
rows removed in the first filtering round: 18
additional rows removed in the second filtering round (whole word left with only one sense): 7
number of headwords removed as whole words: 5
first 10 example headwords removed as whole words: [source-language headwords]
final removed row count: 25
final output row count: 458
original headword count: 133
retained headword count: 128
fully removed headword count: 5
first 10 fully removed headwords: [source-language headwords]

=== additional homonym definition-level filtering summary ===
number of headwords with only one unique definition entry: 3
first 10 example headwords: [source-language headwords]
rows removed because the whole headword had only one unique definition entry: 8
remaining homonym rows after definition-level filtering: 450



## Step 3: Build Data-Preprocess Summary Sheets

This step reads one cleaned lexical table and generates summary sheets for POS counts, sense/example distributions, and rows marked for removal. Run it separately for homonym, polyseme, and monoseme by changing the input and output placeholders.



In [3]:
import os
import pandas as pd
import numpy as np

INPUT_PATH  = r"CHANGE_THIS_TO_YOUR_PATH/other_svaed_new/01_organization/polyseme_pos_sense_example_cleaned.xlsx"
OUTPUT_PATH = r"CHANGE_THIS_TO_YOUR_PATH/other_svaed_new/01_organization/polyseme_data_preprocess.xlsx"


HEADWORD_COL_NAME = ""

REQUIRED_COLS = ["clean1", "pos", "sense1", "example1"]

def read_any(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path.lower())[1]
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(path)
    if ext in (".csv", ".tsv"):
        sep = "," if ext == ".csv" else "\t"
        return pd.read_csv(path, sep=sep, encoding="utf-8-sig")
    raise ValueError("Only .xlsx/.xls/.csv/.tsv input files are supported.")

_BLANK_RE = re.compile(r"^\s*$", flags=re.UNICODE)
def is_blank_cell(x) -> bool:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return True
    s = str(x)
    s = (s.replace("\u00A0", " ")
           .replace("\u3000", " ")
           .replace("\u200B", "")
           .replace("\ufeff", ""))  # BOM
    s = s.strip()
    if s == "" or s.lower() == "nan":
        return True
    return _BLANK_RE.match(s) is not None

# POS parsing and normalization helper.
def get_main_pos(pos_str: str) -> str:
    """
    Return the primary POS category before the first semicolon-like separator.
    """
    if is_blank_cell(pos_str):
        return ""
    s = str(pos_str).strip()
    parts = re.split(r"[；;]\s*", s)
    return parts[0].strip() if parts and parts[0].strip() else ""

def main():
    df = read_any(INPUT_PATH)

    for c in REQUIRED_COLS:
        if c not in df.columns:
            raise KeyError(f"Input file is missing required column: {c}")

    if HEADWORD_COL_NAME and HEADWORD_COL_NAME in df.columns:
        head_col = HEADWORD_COL_NAME
    else:
        head_col = df.columns[1]

    for c in [head_col, "pos", "sense1", "example1"]:
        df[c] = df[c].astype(str).str.strip()

    # POS parsing and normalization helper.
    pos_cat = df[["pos", head_col]].copy()
    pos_cat["_pos_display"] = pos_cat["pos"].where(~pos_cat["pos"].apply(is_blank_cell), "(blank)")

    if pos_cat.empty:
        pos_stats = pd.DataFrame(columns=["pos_category", "pos_number", "pos_words"])
    else:
        grouped = (
            pos_cat.groupby("_pos_display")[head_col]
                   .apply(lambda s: sorted(set(s.dropna())))
                   .reset_index()
        )
        grouped["pos_number"] = grouped[head_col].apply(len)
        grouped["pos_words"]  = grouped[head_col].apply(lambda lst: "、".join(lst))
        pos_stats = (grouped
                     .rename(columns={"_pos_display": "pos_category"})
                     .drop(columns=[head_col])
                     .sort_values(by=["pos_number", "pos_category"], ascending=[False, True])
                     .reset_index(drop=True))

    # Example-sentence processing rule.
    # POS parsing and normalization helper.
    df["_main_pos"] = df["pos"].apply(get_main_pos)

    # POS parsing and normalization helper.
    def build_pos_combination(series: pd.Series) -> str:
        tokens = [p for p in series.tolist() if p]
        if not tokens:
            return "(blank)"
        uniq_sorted = sorted(set(tokens))
        return "+".join(uniq_sorted)

    hw_pos_combo = (df.groupby(head_col)["_main_pos"]
                      .apply(build_pos_combination)
                      .reset_index()
                      .rename(columns={"_main_pos": "pos_combination"}))

    # Sense-level processing rule.
    # Example-sentence processing rule.
    sense_counts = (
        df.loc[~df["sense1"].apply(is_blank_cell)]
          .groupby(head_col)["sense1"].nunique()
          .reindex(df[head_col].unique(), fill_value=0)
          .reset_index().rename(columns={"sense1": "num_senses"})
    )
    example_counts = (
        (~df["example1"].apply(is_blank_cell))
          .groupby(df[head_col]).sum()
          .reindex(df[head_col].unique(), fill_value=0)
          .reset_index().rename(columns={"example1": "num_examples"})
    )
    # POS parsing and normalization helper.
    hw_summary = (sense_counts
                  .merge(example_counts, on=head_col, how="left")
                  .merge(hw_pos_combo, on=head_col, how="left"))

    if hw_summary.empty:
        sense_example_grid = pd.DataFrame(columns=["sense_category_number", "pos_combination", "example_number", "all_number", "all_words"])
    else:
        g2 = (
            hw_summary.groupby(["num_senses", "pos_combination", "num_examples"])[head_col]
                      .apply(lambda s: sorted(set(s)))
                      .reset_index()
        )
        g2["all_number"] = g2[head_col].apply(len)
        g2["all_words"]  = g2[head_col].apply(lambda lst: "、".join(lst))
        sense_example_grid = (g2.rename(columns={
                                "num_senses":     "sense_category_number",
                                "num_examples":   "example_number"
                              })
                              .drop(columns=[head_col])
                              .sort_values(by=["sense_category_number", "pos_combination", "example_number"])
                              .reset_index(drop=True))

    # Example-sentence processing rule.
    #mask_blank = df["example1"].apply(is_blank_cell)
    mask_blank = df["sense1"].str.contains(r"〈方〉", na=False)
    blank_rows = df.loc[mask_blank, [head_col, "sense1"]].copy()
    blank_rows.columns = ["headword", "sense_tagged_to_remove"]
    if not blank_rows.empty:
        #blank_rows["tag_detected"] = "blank"
        blank_rows["tag_detected"]    = "〈方〉"
        blank_rows["total_summary"] = ""
        total_n = blank_rows.shape[0]
        summary_row = pd.DataFrame([{"headword": "", "sense_tagged_to_remove": "", "tag_detected": "", "total_summary": f"Total: {total_n}"}])
        blank_examples = pd.concat([blank_rows, summary_row], ignore_index=True)
    else:
        blank_examples = pd.DataFrame([{"headword": "", "sense_tagged_to_remove": "", "tag_detected": "", "total_summary": "Total: 0"}])

    with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
        pos_stats.to_excel(writer, sheet_name="pos_stats", index=False)
        sense_example_grid.to_excel(writer, sheet_name="sense_example_grid", index=False)
        blank_examples.to_excel(writer, sheet_name="blank_examples", index=False)

    print("Summary completed")
    print(f"Output file: {OUTPUT_PATH}")
    print("\n[Preview] first few rows of sense_example_grid:")
    print(sense_example_grid.head())
    print("\n[Preview] first few rows of pos_stats:")
    print(pos_stats.head())
    print("\n[Preview] last few rows of blank_examples including the total row:")
    print(blank_examples.tail())

if __name__ == "__main__":
    main()


Summary completed
Output file: CHANGE_THIS_TO_YOUR_PATH/other_svaed_new/01_organization/polyseme_data_preprocess.xlsx

[Preview] first few rows of sense_example_grid:
[Preview table omitted here: source-language headwords and POS categories will appear when you run this cell on your data.]

[Preview] first few rows of pos_stats:
[Preview table omitted here: source-language POS labels and word lists will appear when you run this cell on your data.]

[Preview] last few rows of blank_examples including the total row:
Total: 0
